In [1]:
import csv
import json
import time
import urllib.request
import urllib.parse
from datetime import datetime, timedelta, timezone

import pandas as pd

USER_AGENT = "rss-backfill-notebook/1.0 (personal research tool)"

In [2]:
#define parameters for the data collection

SUBREDDIT = "nascar"          # without r/
START_DATE = "2025-02-02"  # YYYY-MM-DD, UTC
END_DATE = "2026-06-30"    # YYYY-MM-DD, UTC
KEYWORD = "FedEx"              # sponsor here
CHUNK_DAYS = 7               # size of each date window; shrink for high-volume subs
DELAY_SECONDS = 8.0          # politeness delay between requests (raise if you still see 429s)
MAX_RETRIES = 5               # retries per chunk on a 429 before giving up on it
BACKOFF_BASE = 15.0           # seconds; used if Arctic Shift doesn't send a Retry-After header
OUTPUT_CSV = "../csvs/raw/reddit_posts.csv"

RESUME_AFTER = None  # int epoch seconds (UTC); set to None to start from START_DATE

In [3]:
ARCTIC_SHIFT_SEARCH_URL = "https://arctic-shift.photon-reddit.com/api/posts/search"
FIELDS = "id,title,selftext,author,created_utc"
RETRYABLE_STATUS_CODES = {429, 422, 502, 503, 504}  # 422 observed as an intermittent backend flake, not a bad request


def build_search_url(subreddit: str, after: int, before: int, limit="auto", sort="asc") -> str:
    params = {
        "subreddit": subreddit,
        "after": after,   # epoch seconds
        "before": before, # epoch seconds
        "sort": sort,
        "limit": limit,   # "auto" lets Arctic Shift return 100-1000 per page
        "fields": FIELDS,
    }
    return f"{ARCTIC_SHIFT_SEARCH_URL}?{urllib.parse.urlencode(params)}"


def fetch_json(url: str, timeout: int = 30) -> list[dict]:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        payload = json.loads(resp.read())
    if "error" in payload:
        raise RuntimeError(payload["error"])
    return payload["data"]


def fetch_json_with_retry(url: str, max_retries: int, backoff_base: float, timeout: int = 30) -> list[dict]:
    """Fetch a page of posts, retrying on rate limits / transient errors with backoff.

    Honors the Retry-After header if Arctic Shift sends one; otherwise backs
    off exponentially starting from backoff_base seconds.
    """
    attempt = 0
    while True:
        try:
            return fetch_json(url, timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code not in RETRYABLE_STATUS_CODES or attempt >= max_retries:
                raise
            retry_after = e.headers.get("Retry-After") if e.headers else None
            if retry_after is not None:
                try:
                    wait = float(retry_after)
                except ValueError:
                    wait = backoff_base * (2 ** attempt)
            else:
                wait = backoff_base * (2 ** attempt)
            attempt += 1
            print(f"    {e.code} error, waiting {wait:.0f}s before retry {attempt}/{max_retries} ... ", end="")
            time.sleep(wait)


def to_entry(item: dict, subreddit: str) -> dict:
    """Convert an Arctic Shift post object into the same shape the old RSS pipeline used."""
    created = datetime.fromtimestamp(item["created_utc"], tz=timezone.utc)
    post_id = item.get("id")
    author = item.get("author") or ""
    return {
        "id": f"t3_{post_id}",
        "title": item.get("title") or "",
        "link": f"https://www.reddit.com/r/{subreddit}/comments/{post_id}/" if post_id else "",
        "author": f"/u/{author}" if author else "",
        "published": created.isoformat(),
        "content": item.get("selftext") or "",
    }

In [4]:
start_ts = int(datetime.strptime(START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
end_ts = int(datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
assert end_ts > start_ts, "END_DATE must be after START_DATE"

seen_ids = set()
all_entries = []
cursor = RESUME_AFTER if RESUME_AFTER is not None else start_ts
page = 0
stopped_early = False

print(f"Backfilling r/{SUBREDDIT} from {START_DATE} to {END_DATE} via Arctic Shift...")

while cursor < end_ts:
    page += 1
    url = build_search_url(SUBREDDIT, after=cursor, before=end_ts)
    print(f"[page {page}] after={datetime.fromtimestamp(cursor, tz=timezone.utc).date()} ... ", end="")

    try:
        items = fetch_json_with_retry(url, MAX_RETRIES, BACKOFF_BASE)
    except Exception as e:
        print(f"FAILED ({e}) — stopping early")
        stopped_early = True
        break

    if not items:
        print("no more results — reached END_DATE")
        break

    new_count = 0
    for item in items:
        entry = to_entry(item, SUBREDDIT)
        if entry["id"] not in seen_ids:
            seen_ids.add(entry["id"])
            all_entries.append(entry)
            new_count += 1

    last_created_ts = items[-1]["created_utc"]
    print(f"{len(items)} entries, {new_count} new (newest on page: {datetime.fromtimestamp(last_created_ts, tz=timezone.utc)})")

    if last_created_ts <= cursor:
        # no forward progress (shouldn't happen with sort=asc) — bail to avoid an infinite loop
        print("cursor did not advance — stopping")
        break

    cursor = last_created_ts + 1
    time.sleep(DELAY_SECONDS)

print(f"\nDone. {len(all_entries)} unique posts collected.")
if stopped_early:
    print(f"Stopped early due to repeated failures. To resume later, set:")
    print(f"    RESUME_AFTER = {cursor}")
    print(f"and rerun from the parameters cell down.")

Backfilling r/nascar from 2025-02-02 to 2026-06-30 via Arctic Shift...
[page 1] after=2025-02-02 ... 100 entries, 100 new (newest on page: 2025-02-03 02:50:55+00:00)
[page 2] after=2025-02-03 ... 100 entries, 100 new (newest on page: 2025-02-04 15:06:32+00:00)
[page 3] after=2025-02-04 ...     422 error, waiting 15s before retry 1/5 ... 100 entries, 100 new (newest on page: 2025-02-06 07:52:07+00:00)
[page 4] after=2025-02-06 ... 100 entries, 100 new (newest on page: 2025-02-07 18:43:01+00:00)
[page 5] after=2025-02-07 ... 100 entries, 100 new (newest on page: 2025-02-09 03:26:57+00:00)
[page 6] after=2025-02-09 ... 100 entries, 100 new (newest on page: 2025-02-10 18:15:55+00:00)
[page 7] after=2025-02-10 ... 100 entries, 100 new (newest on page: 2025-02-11 19:47:00+00:00)
[page 8] after=2025-02-11 ... 100 entries, 100 new (newest on page: 2025-02-12 22:10:33+00:00)
[page 9] after=2025-02-12 ...     422 error, waiting 15s before retry 1/5 ... 100 entries, 100 new (newest on page: 2025-

In [5]:
df = pd.DataFrame(all_entries)
df["published"] = pd.to_datetime(df["published"], errors="coerce")
df = df.sort_values("published").reset_index(drop=True)
df.head()

,id,title,link,author,published,content
0,t3_1ifk7br,NASCAR daily podcast,https://www.reddit.com/r/nascar/comments/1ifk7br/,/u/Blue_Gi11,2025-02-02 00:01:55+00:00,When is it on?
1,t3_1ifklb3,15 Days Until the 67th Daytona 500: Caraway Sp...,https://www.reddit.com/r/nascar/comments/1ifklb3/,/u/bruhmoment2248,2025-02-02 00:21:07+00:00,# The Southern Short Track Racing Icon\n\nWhil...
2,t3_1ifkx1b,Life is good tonight!,https://www.reddit.com/r/nascar/comments/1ifkx1b/,/u/joedidder,2025-02-02 00:37:11+00:00,Cup on the big screen and Lucas Oil Late Model...
3,t3_1ifkxxa,Awesome podcast interviewing old drivers in Mi...,https://www.reddit.com/r/nascar/comments/1ifkxxa/,/u/Disastrous_Video9160,2025-02-02 00:38:27+00:00,[https://www.youtube.com/live/2m6mmfwF2p8](htt...
4,t3_1iflbiz,Informational shit about Michigan Nascar,https://www.reddit.com/r/nascar/comments/1iflbiz/,/u/Disastrous_Video9160,2025-02-02 00:57:33+00:00,I love YouTube shit about cars and I just foun...


In [ ]:
def construct_df_sponsor(df: pd.DataFrame, sponsor: str, driver: str, team: str) -> pd.DataFrame:
    """Construct a DataFrame with posts containing the sponsor keyword."""
    df_filtered = df[df["title"].str.contains(sponsor, case=False, na=False) |
                     df["content"].str.contains(sponsor, case=False, na=False) |
                     df["title"].str.contains(driver, case=False, na=False) |
                     df["content"].str.contains(driver, case=False, na=False) |
                     df["title"].str.contains(team, case=False, na=False) |
                     df["content"].str.contains(team, case=False, na=False)].copy()
    return df_filtered


SPONSORS = [
    ("Progressive", "Denny Hamlin", "Joe Gibbs Racing"),
    ("Busch Light", "Ross Chastain", "Trackhouse Racing"),
    # FIX: was ("Cheddar's Scratch Kitchen", "Austin Hill", "Richard Childress Racing").
    # The Cheddar's Scratch Kitchen Cup Series deal is on Kyle Busch's #8 RCR car —
    # Austin Hill's #33 only carried Cheddar's branding in 1 of 12 tracked races.
    # See Clean_Race_Data.ipynb sponsor_mapping fix for the underlying evidence.
    ("Cheddar\'s Scratch Kitchen", "Kyle Busch", "Richard Childress Racing"),
    ("Love\'s Travel Stops", "Todd Gilliland", "Front Row Motorsports"),
    ("Castrol", "Brad Keselowski", "RFK Racing"),
]

df_sponsor_dict = {}
for sponsor, driver, team in SPONSORS:
    df_sponsor_dict[sponsor] = construct_df_sponsor(df, sponsor, driver, team)
    print(f"Posts mentioning \'{sponsor}\', \'{driver}\', or \'{team}\': {len(df_sponsor_dict[sponsor])}")

In [7]:
for sponsor, sdf in df_sponsor_dict.items():
    sdf["published"] = pd.to_datetime(sdf["published"], errors="coerce")
    sdf["date"] = sdf["published"].dt.date
    sdf.sort_values("published", inplace=True)
    sdf.reset_index(drop=True, inplace=True)

df_sponsor_dict["Progressive"].head()

,id,title,link,author,published,content,date
0,t3_1ifn13b,Arctos Investment Group and Kings Hawaiian to ...,https://www.reddit.com/r/nascar/comments/1ifn13b/,/u/democracywon2024,2025-02-02 02:28:00+00:00,Brought up how there's some interesting connec...,2025-02-02
1,t3_1ig3jry,[Joe Gibbs Racing on YouTube] Showing off thos...,https://www.reddit.com/r/nascar/comments/1ig3jry/,/u/AgentofChaos17,2025-02-02 18:15:06+00:00,,2025-02-02
2,t3_1ihizjw,2025 Joe Gibbs Racing Interstate Batteries Pai...,https://www.reddit.com/r/nascar/comments/1ihizjw/,/u/Dmacthegoat,2025-02-04 14:33:38+00:00,,2025-02-04
3,t3_1ihzno0,12 Days Until the 67th Daytona 500: Greenville...,https://www.reddit.com/r/nascar/comments/1ihzno0/,/u/bruhmoment2248,2025-02-05 02:28:44+00:00,# S1ap's Home Track\n\nTo the land of rice far...,2025-02-05
4,t3_1iide4j,Denny Hamlin’s National Debt Relief Scheme,https://www.reddit.com/r/nascar/comments/1iide4j/,/u/DDowd86,2025-02-05 16:03:28+00:00,,2025-02-05


In [8]:
import os

output_dir = "data/raw"
os.makedirs(output_dir, exist_ok=True)

for sponsor, sdf in df_sponsor_dict.items():
    filename = f"{sponsor}.csv"
    filepath = os.path.join(output_dir, filename)
    sdf.to_csv(filepath, index=False)

In [9]:
combined = []
for sponsor, sdf in df_sponsor_dict.items():
    tmp = sdf.copy()
    tmp["sponsor"] = sponsor
    combined.append(tmp)

df_sponsors_combined = pd.concat(combined, ignore_index=True)
df_sponsors_combined = df_sponsors_combined.sort_values(["sponsor", "published"]).reset_index(drop=True)
df_sponsors_combined.head()

,id,title,link,author,published,content,date,sponsor
0,t3_1ig01dy,NASCAR's version of the Stig?,https://www.reddit.com/r/nascar/comments/1ig01dy/,/u/Retired_Army_Dude,2025-02-02 15:46:39+00:00,So I work in the media... yesterday while walk...,2025-02-02,Busch Light
1,t3_1ig01e9,NASCAR's version of the Stig?,https://www.reddit.com/r/nascar/comments/1ig01e9/,/u/Retired_Army_Dude,2025-02-02 15:46:39+00:00,So I work in the media... yesterday while walk...,2025-02-02,Busch Light
2,t3_1ig7juf,Pre-Race Discussion Thread: NCS Busch Light Cl...,https://www.reddit.com/r/nascar/comments/1ig7juf/,/u/NASCARThreadBot,2025-02-02 21:00:08+00:00,***Busch Light Clash at Bowman Gray Stadium***...,2025-02-02,Busch Light
3,t3_1ige8r4,Ah yes the Ross Chastain clash,https://www.reddit.com/r/nascar/comments/1ige8r4/,/u/Jobodahobo11,2025-02-03 02:08:12+00:00,,2025-02-03,Busch Light
4,t3_1iibgup,Busch Beer has announced that Ross Chastain wi...,https://www.reddit.com/r/nascar/comments/1iibgup/,/u/vpat48,2025-02-05 14:41:39+00:00,,2025-02-05,Busch Light


In [10]:
df_sponsors_combined.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_sponsors_combined)} rows to {OUTPUT_CSV}")

Saved 1884 rows to ../csvs/raw/reddit_posts.csv
